# SVD-Demo am Transformer: Warum LoRA funktioniert

Dieses Notebook zeigt die zentrale didaktische Brücke zwischen **M1 (PCA)** und **M3 (LoRA)**:
Wir laden ein reales Transformer-Modell, extrahieren eine Gewichtsmatrix und zeigen
mittels Singulärwertzerlegung (SVD), dass die für Fine-Tuning relevanten Gewichts-Updates
eine **niedrige intrinsische Dimension** besitzen – also durch eine Rang-r-Approximation
(Low-Rank) effizient darstellbar sind.

**Voraussetzung:** Das Modell muss lokal auf der Festplatte vorhanden sein (z. B. via
`huggingface-cli download Qwen/Qwen3-1.7B-Instruct`).


---

## Zelle 1: Imports und Konfiguration


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM
import pandas as pd

# =============================================================================
# HIER DEN PFAD ZUM LOKALEN MODELL EINTRAGEN
# =============================================================================
# Beispiele:
#   - "Qwen/Qwen3-1.7B-Instruct"          (wird aus HF-Cache geladen)
#   - "C:/models/Qwen3-1.7B-Instruct"     (Windows, lokaler Pfad)
#   - "/opt/models/Qwen3-1.7B-Instruct"   (Linux, lokaler Pfad)
#   - "./models/Llama-3.2-1B-Instruct"    (relativer Pfad)
# =============================================================================

MODEL_PFAD = "Qwen/Qwen3-1.7B-Instruct"


---

## Zelle 2: Modell laden (Read-Only, CPU)

Wir laden das Modell auf die CPU, um unabhängig von der GPU zu sein.
`torch_dtype=torch.float32` ist wichtig, damit die SVD numerisch stabil ist.
`device_map=None` verhindert automatische Device-Zuweisungen.


In [ ]:
print(f"Lade Modell von: {MODEL_PFAD}")
print("Das kann beim ersten Mal einige Minuten dauern...\n")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PFAD,
    torch_dtype=torch.float32,
    device_map=None,         # CPU-Only, keine automatische Aufteilung
    low_cpu_mem_usage=True,  # spart RAM beim Laden
)
model.eval()  # Inference-Modus

# Alle Parameter einfrieren - wir wollen ja nur beobachten, nicht trainieren
for param in model.parameters():
    param.requires_grad = False

print(f"✓ Modell geladen. Parameter: {sum(p.numel() for p in model.parameters()):,}")


---

## Zelle 3: Architektur des Modells inspizieren

Bevor wir eine Matrix extrahieren, schauen wir uns den Aufbau an.
Transformer bestehen aus gestapelten Blöcken mit `self_attn` und `mlp`.


In [ ]:
# Architektur-Übersicht als Tabelle
architektur_info = []

for name, module in model.named_modules():
    if "self_attn" in name or "mlp" in name:
        if hasattr(module, "weight"):
            architektur_info.append({
                "Layer": name,
                "Shape": tuple(module.weight.shape),
                "Parameter": module.weight.numel(),
            })
        # Auch Submodule (q_proj, v_proj, etc.) zeigen
        for sub_name, sub_module in module.named_children():
            if hasattr(sub_module, "weight"):
                architektur_info.append({
                    "Layer": f"{name}.{sub_name}",
                    "Shape": tuple(sub_module.weight.shape),
                    "Parameter": sub_module.weight.numel(),
                })

# Nur die ersten 20 Zeilen für Übersichtlichkeit anzeigen
df = pd.DataFrame(architektur_info)
print("=== Ersten 20 Schichten des Modells ===\n")
print(df.head(20).to_string(index=False))
print(f"\n... insgesamt {len(df)} Layer mit Gewichtsmatrizen")


---

## Zelle 4: Eine konkrete Gewichtsmatrix auswählen

Wir wählen die **Query-Projektion der ersten Attention-Schicht** (`q_proj` in Layer 0).
Diese Matrix ist ein idealer Kandidat, weil LoRA typischerweise auf genau solche
Attention-Projektionen angewendet wird.

**Hinweis:** Die Pfadangabe ist modellspezifisch. Bei Qwen3/Llama3:
`model.model.layers[0].self_attn.q_proj.weight`


In [ ]:
# Matrix extrahieren (als PyTorch-Tensor)
W_tensor = model.model.layers[0].self_attn.q_proj.weight

# In NumPy-Array konvertieren für SVD
W = W_tensor.detach().cpu().numpy()

print(f"Matrix: model.model.layers[0].self_attn.q_proj.weight")
print(f"Shape: {W.shape}  →  (out_features × in_features)")
print(f"Datentyp: {W.dtype}")
print(f"Anzahl Parameter in dieser einen Matrix: {W.size:,}")


---

## Zelle 5: Singulärwertzerlegung (SVD) berechnen

Die SVD zerlegt jede Matrix W in drei Matrizen:
    W = U · Σ · Vᵀ
wobei Σ eine Diagonalmatrix mit den **Singulärwerten** ist.

Diese Singulärwerte sagen uns, wie viel "Informationsgehalt" in jeder
Dimension steckt. Fallen sie schnell ab, ist die Matrix **low-rank** –
genau die Voraussetzung für LoRA.


In [ ]:
print("Berechne SVD... (das dauert bei großen Matrizen ~30 Sekunden)")

# Vollständige SVD
U, S, Vt = np.linalg.svd(W, full_matrices=False)

print(f"✓ SVD abgeschlossen.")
print(f"U-Shape: {U.shape}")
print(f"S-Shape: {S.shape}  (Singulärwerte)")
print(f"Vt-Shape: {Vt.shape}")
print(f"\nGrößte 10 Singulärwerte: {S[:10].round(2)}")
print(f"Kleinste 10 Singulärwerte: {S[-10:].round(4)}")


---

## Zelle 6: Plot 1 – Das Singulärwertspektrum

Dies ist **das** zentrale Bild der didaktischen Einheit.
Ein steiler Abfall der Singulärwerte beweist, dass die Matrix
eine niedrige intrinsische Dimension hat.


In [ ]:
plt.figure(figsize=(12, 6))

# Singulärwerte auf logarithmischer Skala
plt.semilogy(S, 'b-', linewidth=2, label='Singulärwerte')
plt.axhline(y=S[0] * 0.01, color='r', linestyle='--', alpha=0.7,
            label='1% des größten Singulärwerts')

plt.xlabel('Index der Singulärwerte (absteigend sortiert)', fontsize=12)
plt.ylabel('Wert (log. Skala)', fontsize=12)
plt.title(f'Singulärwertspektrum von q_proj (Layer 0)\n'
          f'Matrix-Shape: {W.shape}, {W.size:,} Parameter',
          fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Quantitative Aussage: wie viel Energie steckt in den ersten r Komponenten?
total_energy = np.sum(S**2)
energy_r8 = np.sum(S[:8]**2)
energy_r16 = np.sum(S[:16]**2)

print(f"\nEnergie (Frobenius-Norm²) der Matrix: {total_energy:.2f}")
print(f"  Anteil in Top-8  Singulärwerten:  {energy_r8/total_energy*100:.1f} %")
print(f"  Anteil in Top-16 Singulärwerten:  {energy_r16/total_energy*100:.1f} %")
print(f"\n→ LoRA mit r=8 bis r=16 kann einen Großteil der Matrixstruktur abbilden.")


---

## Zelle 7: Plot 2 – Rekonstruktionsfehler über dem Rang r

Wir approximieren W mit Rang r und messen, wie groß der Fehler ist.
Das ist genau das, was LoRA während des Trainings tut: Es sucht die
besten Matrizen A und B, sodass B·A ≈ ΔW.


In [ ]:
ranks = list(range(1, min(128, len(S))))  # teste Rang 1 bis 128
reconstruction_errors = []

for r in ranks:
    # Rang-r-Approximation: W_r = U[:,:r] @ diag(S[:r]) @ Vt[:r,:]
    W_approx = U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :]

    # Frobenius-Norm des Fehlers
    error = np.linalg.norm(W - W_approx, 'fro')
    reconstruction_errors.append(error)

# Normalisieren auf Fehler bei Rang 1
max_error = reconstruction_errors[0]
relative_errors = [e / max_error for e in reconstruction_errors]

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Links: absoluter Fehler
ax1.plot(ranks, reconstruction_errors, 'g-', linewidth=2)
ax1.set_xlabel('Rang r', fontsize=12)
ax1.set_ylabel('Rekonstruktionsfehler (Frobenius-Norm)', fontsize=12)
ax1.set_title('Absoluter Fehler über Rang r', fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.axvline(x=8, color='orange', linestyle='--', label='r=8 (typisches LoRA)')
ax1.axvline(x=16, color='red', linestyle='--', label='r=16 (typisches LoRA)')
ax1.legend()

# Rechts: relativer Fehler
ax2.plot(ranks, relative_errors, 'g-', linewidth=2)
ax2.set_xlabel('Rang r', fontsize=12)
ax2.set_ylabel('Relativer Fehler (normiert auf r=1)', fontsize=12)
ax2.set_title('Relativer Fehler über Rang r', fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.axvline(x=8, color='orange', linestyle='--', label='r=8')
ax2.axvline(x=16, color='red', linestyle='--', label='r=16')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Rekonstruktionsfehler bei r=8:  {relative_errors[7]*100:.2f} % vom Fehler bei r=1")
print(f"Rekonstruktionsfehler bei r=16: {relative_errors[15]*100:.2f} % vom Fehler bei r=1")


---

## Zelle 8: Visualisierung – Original vs. Rang-8-Approximation

Zum Abschluss sehen wir direkt, wie gut eine Rang-8-Approximation
das Original-Bild der Gewichtsmatrix wiedergibt.
LoRA mit r=8 trainiert genau diese 8 "Hauptkomponenten" des Deltas.


In [ ]:
r = 8
W_r8 = U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Original
im1 = axes[0].imshow(W, cmap='viridis', aspect='auto')
axes[0].set_title(f'Original W\n({W.size:,} Parameter)', fontsize=12)
plt.colorbar(im1, ax=axes[0], fraction=0.046)

# Rang-8-Approximation
im2 = axes[1].imshow(W_r8, cmap='viridis', aspect='auto')
axes[1].set_title(f'Rang-8-Approximation\n({r * (W.shape[0] + W.shape[1]):,} Parameter)',
                  fontsize=12)
plt.colorbar(im2, ax=axes[1], fraction=0.046)

# Differenz
diff = W - W_r8
im3 = axes[2].imshow(diff, cmap='RdBu', aspect='auto',
                     vmin=-np.abs(diff).max(), vmax=np.abs(diff).max())
axes[2].set_title(f'Differenz (Rest-Fehler)\nFrobenius-Norm: {np.linalg.norm(diff, "fro"):.2f}',
                  fontsize=12)
plt.colorbar(im3, ax=axes[2], fraction=0.046)

plt.suptitle('LoRA-Prinzip visualisiert: Wenige Hauptkomponenten dominieren die Matrix',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Speichereinsparung ausrechnen
original_params = W.size
lora_params = r * (W.shape[0] + W.shape[1])  # B (out×r) + A (r×in)
ersparnis = (1 - lora_params / original_params) * 100

print(f"\n=== Parameter-Vergleich für q_proj (Layer 0) ===")
print(f"Original-Matrix:      {original_params:>8,} Parameter")
print(f"LoRA (r={r}):           {lora_params:>8,} Parameter  (A + B)")
print(f"Speichereinsparung:   {ersparnis:>7.1f} %")
print(f"\nDas ist der Grund, warum LoRA-Training auf lokaler Hardware funktioniert.")


---

## Zelle 9: Bonus – Vergleich über mehrere Layer hinweg

Abschließend schauen wir, ob das Low-Rank-Muster in allen Schichten
gleich ausgeprägt ist. Das motiviert, warum LoRA in **allen** Layern
und nicht nur im letzten platziert wird.


In [ ]:
# Alle q_proj-Matrizen aus allen Layern sammeln
q_proj_matrices = []
layer_names = []

for name, module in model.named_modules():
    if name.endswith("self_attn.q_proj"):
        W_layer = module.weight.detach().cpu().numpy()
        q_proj_matrices.append(W_layer)
        layer_names.append(name)

print(f"{len(q_proj_matrices)} q_proj-Matrizen gefunden.\n")

# SVD für jede Matrix berechnen und Spektrum plotten
plt.figure(figsize=(12, 7))

for i, (W_layer, name) in enumerate(zip(q_proj_matrices, layer_names)):
    _, S_layer, _ = np.linalg.svd(W_layer, full_matrices=False)
    # Normieren auf ersten Singulärwert für Vergleichbarkeit
    S_norm = S_layer / S_layer[0]

    # Nur jeden 4. Layer plotten, um das Bild nicht zu überladen
    if i % 4 == 0:
        layer_idx = name.split('.')[2]
        plt.semilogy(S_norm, alpha=0.7, label=f'Layer {layer_idx}')

plt.xlabel('Index der Singulärwerte', fontsize=12)
plt.ylabel('Normierter Singulärwert (log. Skala)', fontsize=12)
plt.title('Singulärwertspektren aller q_proj-Layer im Modell\n'
          '(normiert, jeder 4. Layer dargestellt)', fontsize=13)
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

print("Beobachtung: Das Abklingmuster ist in allen Layern sehr ähnlich.")
print("→ Eine einheitliche LoRA-Konfiguration (gleiches r) funktioniert über alle Schichten.")
